In [4]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parents[0]
DF_INTERIM = ROOT / 'data' / 'interim'

In [11]:
df_objects = pd.read_parquet(DF_INTERIM / 'objects.parquet')
df_objects.head()

,oid,meanra,meandec,ndet,firstmjd,lastmjd,first_observed_at,last_observed_at
0,ZTF17aaaaaak,59.128486,49.427638,1,61229.482338,61229.482338,2026-07-08 11:34:34.003000+00:00,2026-07-08 11:34:34.003000+00:00
1,ZTF17aaaaaal,59.899025,50.278091,284,58365.475035,61254.486435,2018-09-04 11:24:02.998000+00:00,2026-08-02 11:40:28.001000+00:00
2,ZTF17aaaaaam,60.275151,49.811770,182,58441.450301,60966.401991,2018-11-19 10:48:25.998000+00:00,2025-10-18 09:38:51.996000+00:00
3,ZTF17aaaaaan,64.334154,45.594258,588,58343.446620,61256.488137,2018-08-13 10:43:08.003000+00:00,2026-08-04 11:42:55.002000+00:00
4,ZTF17aaaaaao,63.989736,45.430465,428,58340.493391,61254.488009,2018-08-10 11:50:29+00:00,2026-08-02 11:42:44.004000+00:00


In [12]:
df_detection = pd.read_parquet(DF_INTERIM / 'detection' / 'detections_0001.parquet')
df_probability = pd.read_parquet(DF_INTERIM / 'probability' / 'probabilities_0001.parquet')

In [13]:
df_detection.head()

,oid,candid,mjd,fid,magpsf,sigmapsf,isdiffpos,parent_candid,ra,dec,observed_at
0,ZTF17aaaaaas,780139024415010003,58534.139028,1,18.408876,0.135092,-1,805139274415015015,68.578037,49.087102,2019-02-20 03:20:12.002000+00:00
1,ZTF17aaaaaas,996451014415015006,58750.451019,1,19.438852,0.145746,1,1012449364415015008,68.578156,49.087025,2019-09-24 10:49:27.998000+00:00
2,ZTF17aaaaaas,1122203914415020005,58876.203912,1,18.550800,0.081796,-1,1142204954415015008,68.578022,49.087117,2020-01-28 04:53:37.997000+00:00
3,ZTF17aaaaaaw,993493532215010009,58747.493530,1,18.800180,0.097762,-1,1009487112215010011,62.784025,46.340743,2019-09-21 11:50:41.001000+00:00
4,ZTF17aaaaabd,620487892715015024,58374.487893,2,18.885900,0.092108,1,631506042715010002,71.898601,46.312892,2018-09-13 11:42:33.998000+00:00


In [14]:
df_probability.head()

,oid,classifier_name,classifier_version,class_name,probability,ranking
0,ZTF17aaaaaxa,lc_classifier,hierarchical_rf_1.1.0,E,0.281008,2
1,ZTF17aaaaaxa,lc_classifier,hierarchical_rf_1.1.0,RRL,0.027664,6
2,ZTF17aaaadvl,lc_classifier,hierarchical_rf_1.1.0,LPV,0.009580,7
3,ZTF17aaaaanl,lc_classifier,hierarchical_rf_1.1.0,Blazar,0.003100,10
4,ZTF17aaaaeac,lc_classifier,hierarchical_rf_1.1.0,Blazar,0.104160,3


## SQL

In [ ]:
from sqlalchemy import create_engine, text
import pandas as pd

db_name = "db_alerce"
user = "postgres"
password = "..."
host = "localhost"
port = 5432

engine = create_engine(
    f"postgresql+psycopg://{user}:{password}@{host}:{port}/{db_name}"
)

with engine.connect() as conn:
    print(
        conn.execute(
            text("SELECT current_database()")
        ).scalar()
    )

db_alerce


In [2]:
# import objects
df_objects.to_sql(
    name="objects",
    con=engine,
    if_exists="replace",
    index=False
)

NameError: name 'df_objects' is not defined

In [5]:
# import probabilities
files = sorted(Path(ROOT/'data'/'interim'/'probability').glob("*.parquet"))

for i, file in enumerate(files):
    df_tmp = pd.read_parquet(file)

    df_tmp.to_sql(
        name="probabilities",
        con=engine,
        if_exists="replace" if i == 0 else "append",
        index=False
    )

    print(f"Imported {file.name}: {len(df_tmp)} rows")

PendingRollbackError: Can't reconnect until invalid transaction is rolled back.  Please rollback() fully before proceeding (Background on this error at: https://sqlalche.me/e/21/8s2b)

In [20]:
# import detections
files = sorted(Path(ROOT/'data'/'interim'/'detection').glob("*.parquet"))

for i, file in enumerate(files):
    df_tmp = pd.read_parquet(file)

    df_tmp.to_sql(
        name="detections",
        con=engine,
        if_exists="replace" if i == 0 else "append",
        index=False
    )

    print(f"Imported {file.name}: {len(df_tmp)} rows")

Imported detections_0001.parquet: 266935 rows
Imported detections_0002.parquet: 90640 rows
Imported detections_0003.parquet: 158358 rows
Imported detections_0004.parquet: 75306 rows
Imported detections_0005.parquet: 84502 rows
Imported detections_0006.parquet: 6785 rows
Imported detections_0007.parquet: 6549 rows
Imported detections_0008.parquet: 164721 rows
Imported detections_0009.parquet: 149752 rows
Imported detections_0010.parquet: 190795 rows
Imported detections_0011.parquet: 191266 rows
Imported detections_0012.parquet: 237041 rows
Imported detections_0013.parquet: 508824 rows
Imported detections_0014.parquet: 675376 rows
Imported detections_0015.parquet: 561083 rows
Imported detections_0016.parquet: 218858 rows
Imported detections_0017.parquet: 48605 rows
Imported detections_0018.parquet: 205044 rows
Imported detections_0019.parquet: 79742 rows
Imported detections_0020.parquet: 162288 rows
